# 05 · Climate drivers and CH2025 projections of larch autumn phenology

One method throughout: **linear regression with zone fixed effects**, `doy ~ climate features + C(zone)`. The zone terms absorb the fixed differences between zones (elevation, aspect), so the climate slopes come from year-to-year variation within zones.

1. **Which climate factors influence larch autumn phenology, and how?**
   - 3a: the effect of each climate feature on its own (days per °C, per 100 mm, …)
   - 3b: a multiple regression with the best `MAX_FEATURES` features
2. **How do the trend of the observed years and the CH2025 climate scenarios compare as predictions of future yellowing?**
   - 4: both side by side for the whole AOI: the linear trend (as in `04`) extended into the future, and the regression of 3b projected with CH2025 at each warming level, with the leave-one-year-out skill of both

The phenology date to analyse is selectable: `TARGETS` in `config.py`, either a chosen date of `04` (`"start"`, `"peak"`, `"end"`) or one candidate of `SEASON_METHODS` (e.g. `"Whittaker / GRVI / mid"`). Every step runs for each target.

**Climate data:** the MeteoSwiss gridded observations (1 km) per zone, projected with the CH2025 DAILY-GRIDDED scenarios (both from `03`).

- **Reads:**
  - `results/<AOI>/season_selected.csv` and `season_dates.csv` (from `04`)
  - `cache/climate/<AOI>/*_zones.nc` (from `03`)
- **Writes:** `results/<AOI>/climate/`

**Sample size.** Climate varies mainly from year to year, and all zones share the same year, so the effective sample is about the number of years (≈ 10), not the number of zone-years. CIs are therefore clustered by year and the cross-validation leaves out whole years. Treat effects as indicative.

In [ ]:
import textwrap

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from pyproj import Transformer
from scipy.stats import t as t_dist

import importlib
import config, phenology_utils
importlib.reload(config); importlib.reload(phenology_utils)  # pick up edits to config.py without a kernel restart
from config import *
from phenology_utils import load_zone_table

CLIMATE_FILES = {   # source: (observations, scenarios)
    "grid": ("obs_grid_zones.nc", "ch2025_grid_zones.nc"),
}
MAX_FEATURES = 10                               # climate features per multiple regression
SHOW_FEATURES = 10                               # features per target in the figures of 3a: a number or "all"
REFERENCE = "ref91-20"
# Approximate period in which the global warming levels are reached (IPCC AR6 WG1, central estimates; 3 °C only
# under high emissions). ref91-20 is the CH2025 reference period
GWL_YEARS = {"ref91-20": (1991, 2020), "gwl1.5": (2030, 2040), "gwl2.0": (2040, 2060),
             "gwl2.5": (2050, 2080), "gwl3.0": (2060, 2100)}

CLIMATE_OUT_DIR.mkdir(parents=True, exist_ok=True)
zone_table = load_zone_table()
ZONES = [z for z in zone_table.index if z != "all"]
ZONE_SHARE = zone_table.loc[ZONES, "n_pixels"] / zone_table.loc[ZONES, "n_pixels"].sum()   # for AOI means
DOYS = np.arange(1, 366)
print(f"AOI: {AOI_NAME} | years {FIRST_YEAR}-{LAST_YEAR} | targets: {TARGETS}")

## 1. Phenology dates (targets)

`"start"`, `"peak"` and `"end"` are the chosen dates of `04` (`SEASON_CHOICE`, complete years only); a candidate such as `"Whittaker / GRVI / mid"` is taken from `season_dates.csv`. Models use the topographic zones; `"all"` is only shown for reference.

In [ ]:
def load_targets():
    # "start" / "peak" / "end": the chosen dates of 04; "<method> / <index> / <metric>": one candidate of 04
    selected = pd.read_csv(OUT_DIR / "season_selected.csv")
    dates = pd.read_csv(OUT_DIR / "season_dates.csv")
    parts = []
    for target in TARGETS:
        if target in ("start", "peak", "end"):
            d = selected.rename(columns={target: "doy"})
        else:
            key = [part.strip() for part in target.split("/")]
            d = (dates[(dates.method == key[0]) & (dates["index"] == key[1]) & (dates.metric == key[2])]
                 if len(key) == 3 else dates.iloc[:0])
            if d.empty:
                options = sorted(set(dates.method + " / " + dates["index"] + " / " + dates.metric))
                raise ValueError(f"TARGETS: {target!r} is not in season_dates.csv. Options: start, peak, end, {options}")
        parts.append(d[["zone", "year", "doy"]].dropna().assign(target=target))
    out = pd.concat(parts, ignore_index=True)
    return out[out.year.between(FIRST_YEAR, LAST_YEAR)]


targets = load_targets()
display(targets[targets.zone != "all"].pivot_table(index="target", columns="year", values="doy", aggfunc="size")
        .fillna(0).astype(int).rename_axis(columns="zones per year"))

fig, axes = plt.subplots(1, len(TARGETS), figsize=(4 * len(TARGETS), 3.5), sharey=True, squeeze=False)
for ax, target in zip(axes[0], TARGETS):
    d = targets[targets.target == target]
    for zone, g in d[d.zone != "all"].groupby("zone"):
        ax.plot(g.year, g.doy, "-", lw=0.8, alpha=0.5)
    a = d[d.zone == "all"]
    ax.plot(a.year, a.doy, "ko-", lw=2, label="whole AOI")
    ax.set_title(target)
    ax.set_xlabel("year")
    ax.grid(alpha=0.3)
axes[0, 0].set_ylabel("day of year")
axes[0, 0].legend()
plt.suptitle("Phenology dates per zone (thin) and for the whole AOI")
plt.tight_layout()
plt.show()

## 2. Climate data and features per zone and year

The daily zone series of `03` are arranged as (zone, year, day of year) with 365 days; 29 February is dropped. Model years of CH2025 are numbered 1–30 per warming level.

The **features** are seasonal summaries of each year. They are computed the same way for observations and scenarios. Relative sunshine duration is in the observations only, so it can explain the dates but is not used for the projections. The windows end by mid October so that they precede most transition dates. Snow is not in the scenarios; winter precipitation serves as a snowpack proxy. `used in` names a phenology study that uses the feature, or a close equivalent, as a driver (references at the end).

In [ ]:
def to_matrix(ds):
    # (..., time) -> (..., year, doy); 29 Feb dropped so every year has 365 days
    leap_day = (ds.month == 2) & (ds.doy == 60)
    ds = ds.isel(time=np.flatnonzero(~leap_day.values)).drop_vars("month")
    return ds.set_index(time=["year", "doy"]).unstack("time").reindex(doy=DOYS)


climate = {}
for source, (obs_file, scen_file) in CLIMATE_FILES.items():
    if not (CLIMATE_DIR / obs_file).exists():
        print(f"{source}: {obs_file} missing (run 03_download_climate), source skipped")
        continue
    obs = to_matrix(xr.open_dataset(CLIMATE_DIR / obs_file).load())
    scen = None
    if (CLIMATE_DIR / scen_file).exists():
        scen = to_matrix(xr.open_dataset(CLIMATE_DIR / scen_file).load())
    else:
        print(f"{source}: {scen_file} missing, no projections for this source")
    climate[source] = {"obs": obs.sel(zone=ZONES), "scen": None if scen is None else scen.sel(zone=ZONES)}
    print(f"{source}: observations {list(obs.data_vars)}, years {int(obs.year.min())}-{int(obs.year.max())}"
          + ("" if scen is None else f" | scenarios {dict(scen.sizes)}"))
if not climate:
    raise FileNotFoundError(f"No climate data in {CLIMATE_DIR}; run 03_download_climate first")

cells = pd.read_csv(CLIMATE_DIR / "zone_cells.csv")
_, LATITUDE = Transformer.from_crs("EPSG:2056", "EPSG:4326", always_xy=True).transform(cells.E.mean(), cells.N.mean())

In [ ]:
def extraterrestrial_radiation(lat, doy):
    # FAO-56 eq. 21, MJ m-2 day-1
    phi = np.radians(lat)
    dr = 1 + 0.033 * np.cos(2 * np.pi * doy / 365)
    delta = 0.409 * np.sin(2 * np.pi * doy / 365 - 1.39)
    ws = np.arccos(-np.tan(phi) * np.tan(delta))
    return 24 * 60 / np.pi * 0.0820 * dr * (ws * np.sin(phi) * np.sin(delta) + np.cos(phi) * np.cos(delta) * np.sin(ws))


RA = xr.DataArray(extraterrestrial_radiation(LATITUDE, DOYS), dims="doy", coords={"doy": DOYS})


def season(da, first, last, how="mean"):
    # Mean or sum over DOY first..last of every year; NaN if a day is missing
    win = da.sel(doy=slice(first, last))
    return win.mean("doy", skipna=False) if how == "mean" else win.sum("doy", skipna=False)


def winter_sum(da):
    # 1 Nov of the previous year - 30 Apr
    return season(da, 305, 365, "sum").shift(year=1) + season(da, 1, 120, "sum")


def pet_hargreaves(ds):
    # mm/day (Hargreaves & Samani 1985)
    return 0.0023 * 0.408 * RA * (ds.tas + 17.8) * np.sqrt((ds.tasmax - ds.tasmin).clip(min=0))


# name: (description, unit, required variables, function of the daily dataset)
FEATURES = {
    "T_spring": ("mean temperature Mar-May", "°C", ["tas"], lambda d: season(d.tas, 60, 151)),
    "GDD5_JanJul": ("growing degree-days above 5 °C, Jan-Jul", "°C d", ["tas"],
                    lambda d: season((d.tas - 5).clip(min=0), 1, 212, "sum")),
    "T_summer": ("mean temperature Jun-Aug", "°C", ["tas"], lambda d: season(d.tas, 152, 243)),
    "T_Sep": ("mean temperature Sep", "°C", ["tas"], lambda d: season(d.tas, 244, 273)),
    "T_Oct1": ("mean temperature 1-15 Oct", "°C", ["tas"], lambda d: season(d.tas, 274, 288)),
    "Tmin_Sep": ("mean minimum temperature Sep", "°C", ["tasmin"], lambda d: season(d.tasmin, 244, 273)),
    "frost_days": ("days with Tmin < 0 °C, 15 Aug-15 Oct", "days", ["tasmin"],
                   lambda d: season((d.tasmin < 0).where(d.tasmin.notnull()), 227, 288, "sum")),
    "CDD10_AugSep": ("cold degree-days below 10 °C, Aug-Sep", "°C d", ["tas"],
                     lambda d: season((10 - d.tas).clip(min=0), 213, 273, "sum")),
    "P_winter": ("precipitation Nov-Apr (snowpack proxy)", "mm", ["pr"], lambda d: winter_sum(d.pr)),
    "P_summer": ("precipitation Jun-Aug", "mm", ["pr"], lambda d: season(d.pr, 152, 243, "sum")),
    "P_AugSep": ("precipitation Aug-Sep", "mm", ["pr"], lambda d: season(d.pr, 213, 273, "sum")),
    "CWB_summer": ("climatic water balance P - PET (Hargreaves), Jun-Sep", "mm", ["pr", "tas", "tasmin", "tasmax"],
                   lambda d: season(d.pr - pet_hargreaves(d), 152, 273, "sum")),
    "sunshine_AugSep": ("relative sunshine duration Aug-Sep", "%", ["srel"], lambda d: season(d.srel, 213, 273)),
}


# A phenology study using each feature (or a close equivalent) as a driver; full references at the end
FEATURE_SOURCES = {
    "T_spring": "Busetto et al. 2010: spring temperature, larch in the Alps",
    "GDD5_JanJul": "Bigler & Vitasse 2021: growing degree-days May-Jun",
    "T_summer": "Estrella & Menzel 2006: monthly temperatures May-Sep",
    "T_Sep": "Busetto et al. 2010: September temperature, larch in the Alps",
    "T_Oct1": "Liu et al. 2016: preseason temperature",
    "Tmin_Sep": "Wu et al. 2018: preseason minimum (night-time) temperature",
    "frost_days": "Bigler & Vitasse 2021: cold spells in early autumn",
    "CDD10_AugSep": "Delpierre et al. 2009: cold degree-days",
    "P_winter": "Asam et al. 2018: snow cover duration, Alps",
    "P_summer": "Liu et al. 2016: preseason precipitation",
    "P_AugSep": "Xie et al. 2018: heavy rainfall",
    "CWB_summer": "Bigler & Vitasse 2021: climatic water balance Jun",
    "sunshine_AugSep": "Liu et al. 2016: insolation",
}


def compute_features(ds):
    return xr.Dataset({name: f(ds) for name, (_, _, needs, f) in FEATURES.items() if set(needs) <= set(ds.data_vars)})


info_rows, obs_features = [], {}
for source, c in climate.items():
    feats = compute_features(c["obs"])
    obs_features[source] = feats.to_dataframe().reset_index()
    c["features_scen"] = None if c["scen"] is None else compute_features(c["scen"])
    for name in feats.data_vars:
        info_rows.append({"source": source, "feature": name, "description": FEATURES[name][0],
                          "unit": FEATURES[name][1], "used in": FEATURE_SOURCES.get(name, ""),
                          "in_scenarios": c["features_scen"] is not None and name in c["features_scen"]})
feature_info = pd.DataFrame(info_rows)
display(feature_info.pivot_table(index=["feature", "description", "unit", "used in"], columns="source", values="in_scenarios",
                                 aggfunc="first").replace({True: "obs + scenarios", False: "obs only"}))

PLOT_SOURCE = "grid" if "grid" in climate else next(iter(climate))   # source of the partial plots


def model_data(target, source):
    # One row per zone-year: phenology date + features
    d = targets[(targets.target == target) & targets.zone.isin(ZONES)]
    return d.merge(obs_features[source], on=["zone", "year"], how="inner").reset_index(drop=True)


def candidate_features(source, d):
    # Features usable in a model: in the observations and, if the source has scenarios, in the scenarios too (the
    # models are used for projection); at least 80 % of the zone-years must have a value
    c = climate[source]
    feats = [f for f in obs_features[source].columns.drop(["zone", "year"])
             if c["features_scen"] is None or f in c["features_scen"]]
    return [f for f in feats if d[f].notna().mean() > 0.8]


## 3. Q1: Which climate factors matter, and how?

All models are OLS with zone fixed effects, `doy ~ features + C(zone)`, fitted to the zone-years. Equivalently, date and features are centred on each zone's mean and regressed on each other, so elevation differences between zones cannot pose as climate effects.

- **`slope`:** change in date per unit of the feature. CIs are clustered by year, using t quantiles with (years − 1) degrees of freedom.
- **`r2_within`:** the share of the within-zone (year-to-year) variation explained.

### 3a. Effect of each climate feature

One regression per target and feature. **`space_for_time`** is the slope across the zone means (the elevation gradient), for comparison: if it is similar to the slope over the years, warmer sites behave like warmer years; if not, other site factors play a role. It is empty for features that are the same in every zone.

**Figures:** first the strongest features per target (smallest p; `SHOW_FEATURES` of them, or `"all"`) with their direction, the same for the peak alone in landscape format, then the partial plots of the same features against the data. In the first figure the bar is the effect of a typical year, slope × the standard deviation of the feature's year-to-year (within-zone) variation, so features with different units can be compared; whiskers = CI. Red: a higher value delays the date; blue: it advances it; pale: p ≥ `ALPHA`.

In [ ]:
def demean(d, cols):
    return d[cols] - d.groupby("zone")[cols].transform("mean")


def fe_ols(d, feats):
    # OLS doy ~ feats + C(zone) on the zone-demeaned data; cluster-robust (by year) CIs. None if not estimable
    d = d.dropna(subset=feats + ["doy"])
    n_years = d.year.nunique()
    X, y = demean(d, feats).values, demean(d, ["doy"])["doy"].values
    if n_years < 4 or np.linalg.matrix_rank(X) < len(feats):
        return None
    xtx_inv = np.linalg.inv(X.T @ X)
    beta = xtx_inv @ X.T @ y
    resid = y - X @ beta
    score = pd.DataFrame(X * resid[:, None]).groupby(d.year.values).sum().values
    se = np.sqrt(np.diag(n_years / (n_years - 1) * xtx_inv @ (score.T @ score) @ xtx_inv))
    q = t_dist.ppf(1 - ALPHA / 2, n_years - 1)
    coef = pd.DataFrame({"slope": beta, "ci_low": beta - q * se, "ci_high": beta + q * se,
                         "p": 2 * t_dist.sf(np.abs(beta / se), n_years - 1)}, index=feats)
    return {"coef": coef, "r2_within": 1 - (resid @ resid) / (y @ y), "n_zone_years": len(d), "n_years": n_years}


effect_rows = []
for source in climate:
    for target in TARGETS:
        d = model_data(target, source)
        for feature in obs_features[source].columns.drop(["zone", "year"]):
            fit = fe_ols(d, [feature])
            if fit is None:
                continue
            zone_means = d.dropna(subset=[feature]).groupby("zone")[[feature, "doy"]].mean()
            spread = zone_means[feature].max() - zone_means[feature].min()   # 0 if the feature is the same in all zones
            space = (np.polyfit(zone_means[feature], zone_means["doy"], 1)[0]
                     if len(zone_means) >= 3 and spread > 1e-6 * max(1, zone_means[feature].abs().max()) else np.nan)
            effect_rows.append({"source": source, "target": target, "feature": feature,
                                "unit": f"days per {FEATURES[feature][1]}", **fit["coef"].iloc[0].to_dict(),
                                "r2_within": fit["r2_within"], "space_for_time": space,
                                "n_zone_years": fit["n_zone_years"], "n_years": fit["n_years"]})
climate_effects = pd.DataFrame(effect_rows)
for (source, target), g in climate_effects.groupby(["source", "target"], sort=False):
    print(f"{target} ({source} observations)")
    display(g.drop(columns=["source", "target"]).sort_values("p").set_index("feature").round(3))

In [ ]:
def top_features(target):
    # Features of 3a for one target, strongest first (smallest p): SHOW_FEATURES of them, or all
    e = climate_effects[(climate_effects.source == PLOT_SOURCE) & (climate_effects.target == target)].sort_values("p")
    return e if SHOW_FEATURES == "all" else e.head(SHOW_FEATURES)


# The strongest features per target and their direction
DIRECTION_COLORS = {"later": "firebrick", "earlier": "steelblue"}
n_rows = max(len(top_features(target)) for target in TARGETS)
fig, axes = plt.subplots(1, len(TARGETS), figsize=(5.5 * len(TARGETS), 0.8 * n_rows + 1.4), sharex=True,
                         squeeze=False)
for ax, target in zip(axes[0], TARGETS):
    d = model_data(target, PLOT_SOURCE)
    top = top_features(target)
    labels = []
    for k, row in enumerate(top.itertuples()):
        sd = demean(d.dropna(subset=[row.feature]), [row.feature])[row.feature].std()   # typical year-to-year change
        direction = "later" if row.slope > 0 else "earlier"
        ax.barh(k, row.slope * sd, color=DIRECTION_COLORS[direction], alpha=0.8 if row.p < ALPHA else 0.3)
        ax.errorbar(row.slope * sd, k, xerr=[[(row.slope - row.ci_low) * sd], [(row.ci_high - row.slope) * sd]],
                    color="black", capsize=3, lw=1)
        labels.append(f"{row.feature}\n{FEATURES[row.feature][0]}\nhigher → {direction}: "
                      f"{row.slope:+.2f} d per {FEATURES[row.feature][1]}, p = {row.p:.3f}")
    ax.set_yticks(range(len(top)), labels, fontsize=7)
    ax.set_ylim(n_rows - 0.5, -0.5)
    ax.axvline(0, color="black", lw=0.8)
    ax.set_title(target)
    ax.set_xlabel("days per 1 sd of the feature")
    ax.grid(alpha=0.3, axis="x")
handles = [plt.Rectangle((0, 0), 1, 1, color=c, alpha=0.8, label=f"higher value → {k} date")
           for k, c in DIRECTION_COLORS.items()]
handles.append(plt.Rectangle((0, 0), 1, 1, color="grey", alpha=0.3, label=f"p ≥ {ALPHA}"))
fig.legend(handles=handles, loc="lower center", ncol=len(handles), fontsize=8)
shown = "all features" if SHOW_FEATURES == "all" else f"strongest {SHOW_FEATURES}"
fig.suptitle(f"Climate predictors per date type, {shown} ({PLOT_SOURCE} observations, one feature at a time)")
plt.tight_layout(rect=(0, 0.6 / (0.8 * n_rows + 1.4), 1, 1))
plt.show()

In [ ]:
# The same for the peak alone, landscape: the features side by side, strongest on the left, with plain names
PLAIN_NAMES = {
    "T_spring": "Spring temperature\n(Mar–May)",
    "GDD5_JanJul": "Warmth Jan–Jul\n(degree-days > 5 °C)",
    "T_summer": "Summer temperature\n(Jun–Aug)",
    "T_Sep": "September\ntemperature",
    "T_Oct1": "Early-October\ntemperature (1–15 Oct)",
    "Tmin_Sep": "September night\ntemperature",
    "frost_days": "Frost days\n(15 Aug–15 Oct)",
    "CDD10_AugSep": "Cold late summer\n(degree-days < 10 °C,\nAug–Sep)",
    "P_winter": "Winter snow and rain\n(Nov–Apr)",
    "P_summer": "Summer rain\n(Jun–Aug)",
    "P_AugSep": "Late-summer rain\n(Aug–Sep)",
    "CWB_summer": "Summer water balance\n(rain − evaporation,\nJun–Sep)",
    "sunshine_AugSep": "Late-summer sunshine\n(Aug–Sep)",
}
if "peak" in TARGETS:
    d = model_data("peak", PLOT_SOURCE)
    top = top_features("peak")
    fig, ax = plt.subplots(figsize=(1.6 * len(top) + 1.5, 6))
    labels = []
    for k, row in enumerate(top.itertuples()):
        sd = demean(d.dropna(subset=[row.feature]), [row.feature])[row.feature].std()   # typical year-to-year change
        direction = "later" if row.slope > 0 else "earlier"
        ax.bar(k, row.slope * sd, color=DIRECTION_COLORS[direction], alpha=0.8 if row.p < ALPHA else 0.3)
        ax.errorbar(k, row.slope * sd, yerr=[[(row.slope - row.ci_low) * sd], [(row.ci_high - row.slope) * sd]],
                    color="black", capsize=3, lw=1)
        name = PLAIN_NAMES.get(row.feature, textwrap.fill(FEATURES[row.feature][0], 20))
        labels.append(f"{name}\n\n{row.slope:+.2f} days per {FEATURES[row.feature][1]}\np = {row.p:.3f}")
    ax.set_xticks(range(len(top)), labels, fontsize=8)
    ax.set_xlim(-0.6, len(top) - 0.4)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_ylabel("shift of the peak (days)\nfor a typical year-to-year change")
    ax.grid(alpha=0.3, axis="y")
    ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.42), ncol=len(handles), fontsize=8)
    ax.set_title(f"Peak of the yellowing: the {len(top)} strongest climate drivers ({PLOT_SOURCE} observations)")
    plt.tight_layout()
    plt.show()


In [ ]:
# Partial plots of the features of the figure above (within-zone deviations, coloured by year)
N_COLS = 4
for target in TARGETS:
    d = model_data(target, PLOT_SOURCE)
    top = top_features(target)
    if top.empty:
        continue
    ncols = min(len(top), N_COLS)
    nrows = -(-len(top) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.4 * nrows), squeeze=False, layout="constrained")
    for ax, row in zip(axes.flat, top.itertuples()):
        dd = d.dropna(subset=[row.feature])
        x, y = demean(dd, [row.feature])[row.feature], demean(dd, ["doy"])["doy"]
        sc = ax.scatter(x, y, c=dd.year, cmap="viridis", s=18)
        xx = np.array([x.min(), x.max()])
        ax.plot(xx, row.slope * xx, "k-")
        ax.set_xlabel(f"{row.feature} deviation ({FEATURES[row.feature][1]})")
        ax.set_title(f"{row.slope:+.2f} d per {FEATURES[row.feature][1]} (p={row.p:.3f})", fontsize=10)
        ax.grid(alpha=0.3)
    for ax in axes.flat[len(top):]:
        ax.set_visible(False)
    for ax in axes[:, 0]:
        ax.set_ylabel("date deviation (days)")
    fig.colorbar(sc, ax=axes, label="year")
    plt.suptitle(f"{target}: strongest climate effects ({PLOT_SOURCE} observations)")
    plt.show()

### 3b. Multiple regression

Up to `MAX_FEATURES` features are combined in one model per target. They are chosen by forward selection: at each step the feature that most lowers the leave-one-year-out RMSE is added, as long as it lowers it. Candidates are the features that also exist in the CH2025 scenarios (`candidate_features`), because this model is the one projected in 4.

The table shows the final model, fitted on all years. The p-values do not account for the selection and are therefore optimistic; section 4 gives the honest out-of-sample test.

**Figures:** first the features of each final model, drawn as in 3a (bar = slope × the standard deviation of the feature's within-zone variation, whiskers = CI, colour = direction, pale: p ≥ `ALPHA`), but each slope is now the effect with the other features of the model held fixed; the title gives `r2_within`. Then the observed and the fitted dates per year for the whole AOI (pixel-weighted mean of the zones), which shows how much of the year-to-year variation the model reproduces. The fit uses all years, so it is in-sample.

In [ ]:
def fe_fit(d, feats):
    # OLS with zone fixed effects as dummies, for prediction: doy ~ feats + zone
    zones = pd.get_dummies(d.zone, dtype=float)
    X = np.column_stack([d[feats].values, zones.values]) if feats else zones.values
    beta = np.linalg.lstsq(X, d.doy.values, rcond=None)[0]
    return {"features": list(feats), "slopes": beta[:len(feats)], "zones": pd.Series(beta[len(feats):], zones.columns)}


def fe_predict(model, d):
    base = d.zone.map(model["zones"]).values
    return base + (d[model["features"]].values @ model["slopes"] if model["features"] else 0)


def loyo_rmse(d, feats):
    err = []
    for year in d.year.unique():
        tr, te = d[d.year != year], d[d.year == year]
        err.append(fe_predict(fe_fit(tr, feats), te) - te.doy.values)
    err = np.concatenate(err)
    return np.sqrt(np.nanmean(err ** 2))


def select_features(d, candidates):
    chosen, best = [], loyo_rmse(d, [])
    while len(chosen) < MAX_FEATURES:
        scores = {f: loyo_rmse(d, chosen + [f]) for f in candidates if f not in chosen}
        if not scores:
            break
        f, s = min(scores.items(), key=lambda kv: kv[1])
        if s >= best:
            break
        chosen.append(f)
        best = s
    return chosen


def regression_data(target, source):
    # Zone-years of one target with all candidate features (complete rows), and the candidates
    d = model_data(target, source)
    feats = candidate_features(source, d)
    return d.dropna(subset=feats).reset_index(drop=True), feats


final_models, reg_rows = {}, []
for source in climate:
    for target in TARGETS:
        d, feats = regression_data(target, source)
        if d.year.nunique() < 4:
            print(f"{target} ({source}): fewer than 4 years, skipped")
            continue
        chosen = select_features(d, feats)
        final_models[(source, target)] = {"data": d, "model": fe_fit(d, chosen)}
        fit = fe_ols(d, chosen) if chosen else None
        if fit is None:
            reg_rows.append({"source": source, "target": target, "feature": "(none: no feature beats the zone means)"})
            continue
        for feature, c in fit["coef"].iterrows():
            reg_rows.append({"source": source, "target": target, "feature": feature,
                             "unit": f"days per {FEATURES[feature][1]}", **c.to_dict(),
                             "r2_within": fit["r2_within"], "n_zone_years": fit["n_zone_years"],
                             "n_years": fit["n_years"]})
climate_regression = pd.DataFrame(reg_rows)
display(climate_regression.set_index(["source", "target", "feature"]).round(3))

In [ ]:
# The final models: effect of each feature in the model (as in 3a, but with the other feature held fixed)
SOURCE_COLORS = {s: plt.get_cmap("tab10")(k + 1) for k, s in enumerate(climate)}
coefs = climate_regression.dropna(subset=["slope"]) if "slope" in climate_regression else climate_regression.iloc[:0]
n_rows = max([(coefs.target == target).sum() for target in TARGETS] + [1])
fig, axes = plt.subplots(1, len(TARGETS), figsize=(5.5 * len(TARGETS), 0.8 * n_rows + 1.8), sharex=True,
                         squeeze=False)
for ax, target in zip(axes[0], TARGETS):
    c = coefs[coefs.target == target]
    labels = []
    for k, row in enumerate(c.itertuples()):
        d = final_models[(row.source, target)]["data"]
        sd = demean(d, [row.feature])[row.feature].std()   # typical year-to-year change
        direction = "later" if row.slope > 0 else "earlier"
        ax.barh(k, row.slope * sd, color=DIRECTION_COLORS[direction], alpha=0.8 if row.p < ALPHA else 0.3)
        ax.errorbar(row.slope * sd, k, xerr=[[(row.slope - row.ci_low) * sd], [(row.ci_high - row.slope) * sd]],
                    color="black", capsize=3, lw=1)
        labels.append(f"{row.feature}\n{FEATURES[row.feature][0]}\n"
                      f"{row.slope:+.2f} d per {FEATURES[row.feature][1]}, p = {row.p:.3f}")
    ax.set_yticks(range(len(c)), labels, fontsize=7)
    ax.set_ylim(n_rows - 0.5, -0.5)
    if c.empty:
        ax.text(0.5, 0.5, "no feature beats the zone means", ha="center", va="center", transform=ax.transAxes)
    r2 = c.groupby("source", sort=False).r2_within.first()
    ax.set_title(f"{target}\n" + "; ".join(f"R² within = {v:.2f}" for v in r2), fontsize=10)
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("days per 1 sd of the feature")
    ax.grid(alpha=0.3, axis="x")
handles = [plt.Rectangle((0, 0), 1, 1, color=c, alpha=0.8, label=f"higher value → {k} date")
           for k, c in DIRECTION_COLORS.items()]
handles.append(plt.Rectangle((0, 0), 1, 1, color="grey", alpha=0.3, label=f"p ≥ {ALPHA}"))
fig.legend(handles=handles, loc="lower center", ncol=len(handles), fontsize=8)
fig.suptitle("Multiple regression: features of the final model per date type")
plt.tight_layout(rect=(0, 0.4 / (0.8 * n_rows + 1.8), 1, 1))
plt.show()


def aoi_mean(d, values):
    # Pixel-weighted mean over the zones with a value, per year
    ok = np.isfinite(values)
    w = d.zone.map(ZONE_SHARE).values[ok]
    g = pd.DataFrame({"year": d.year.values[ok], "wv": w * values[ok], "w": w}).groupby("year").sum()
    return g.wv / g.w


# Observed and fitted whole-AOI dates per year (fitted on all years, so in-sample; the out-of-sample skill is in 4)
fig, axes = plt.subplots(1, len(TARGETS), figsize=(5.5 * len(TARGETS), 3.8), squeeze=False)
for ax, target in zip(axes[0], TARGETS):
    obs = targets[(targets.target == target) & targets.zone.isin(ZONES)]
    o = aoi_mean(obs, obs.doy.values)
    ax.plot(o.index, o.values, "ko-", ms=4, lw=1.5, label="observed")
    for source in climate:
        if (source, target) not in final_models:
            continue
        model = final_models[(source, target)]["model"]
        d = model_data(target, source).dropna(subset=model["features"])
        f = aoi_mean(d, fe_predict(model, d))
        ax.plot(f.index, f.values, "o--", ms=4, lw=1.5, color=SOURCE_COLORS[source],
                label=f"fitted: {' + '.join(model['features']) or 'zone means only'}")
    ax.set_title(target)
    ax.set_xlabel("year")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7)
axes[0, 0].set_ylabel("day of year (whole AOI)")
fig.suptitle("Multiple regression: observed vs fitted dates (pixel-weighted mean of the zones)")
plt.tight_layout()
plt.show()

## 4. Q2: How do the trend and the CH2025 climate projection compare?

Two predictions of the future whole-AOI date, side by side:

| | Trend | Climate regression + CH2025 |
|---|---|---|
| Model | linear trend of the whole-AOI dates over the observed years (as in `04`) | the regression of 3b (zone means + up to `MAX_FEATURES` climate features), fitted on all years |
| Future | the trend line extended; band = its 95 % CI | applied to every CH2025 model chain and model year of a warming level; 30-year mean per chain, whole AOI = pixel-weighted mean of the zones; point = median, bar = 5–95 % of the chains |

- **`skill`** = 1 − RMSE / RMSE(zone means only), in leave-one-year-out cross-validation on the zone-years (for the regression the feature selection is repeated in every fold). Above 0 = better than no model. A projection whose model has no skill should not be read as a prediction.
- **Timing:** CH2025 gives warming levels, not years. The figure places each level at the middle of the period in which it is reached (`GWL_YEARS`, IPCC AR6 central estimates; 3 °C only under high emissions); the table gives the trend at that year. `ref91-20` is the CH2025 reference period 1991–2020.
- **Extrapolation:** the further the warming, the more scenario years lie outside the observed range of the features (`extrapolation.csv`), where the regression is a straight-line guess. The trend extrapolates in time in the same way; its CI widens quickly and leaves the figure.

In [ ]:
def loyo_predict(d, features):
    # Leave-one-year-out predictions; features(train) -> the features of the model fitted on the training years
    pred = np.full(len(d), np.nan)
    for year in d.year.unique():
        tr, te = (d.year != year).values, (d.year == year).values
        pred[te] = fe_predict(fe_fit(d[tr], features(d[tr])), d[te])
    return pred


def cv_row(target, approach, d, pred):
    rmse = lambda p: np.sqrt(np.nanmean((p - d.doy.values) ** 2))
    return {"target": target, "approach": approach, "cv_rmse": rmse(pred),
            "skill": 1 - rmse(pred) / rmse(loyo_predict(d, lambda tr: []))}


def trend_line(target, years):
    # OLS of the whole-AOI dates on the year (as in 04): the line and its 95 % CI at `years`
    a = targets[(targets.target == target) & (targets.zone == "all")]
    x, y = a.year.values.astype(float), a.doy.values
    slope, intercept = np.polyfit(x, y, 1)
    s = np.sqrt(((y - intercept - slope * x) ** 2).sum() / (len(x) - 2))
    half = (t_dist.ppf(1 - ALPHA / 2, len(x) - 2) * s
            * np.sqrt(1 / len(x) + (years - x.mean()) ** 2 / ((x - x.mean()) ** 2).sum()))
    fit = intercept + slope * years
    return fit, fit - half, fit + half


def regression_projection(model, feats_scen):
    # (period, model, zone, year) predicted DOY
    zone_eff = xr.DataArray(model["zones"].reindex(ZONES).values, dims="zone", coords={"zone": ZONES})
    out = zone_eff + 0 * feats_scen[list(feats_scen.data_vars)[0]]
    for f, b in zip(model["features"], model["slopes"]):
        out = out + b * feats_scen[f]
    return out


PERIODS = [p for p in CH2025_PERIODS if p in GWL_YEARS]
MID_YEAR = {p: sum(GWL_YEARS[p]) / 2 for p in PERIODS}

# ---- leave-one-year-out skill: trend on all zone-years, climate regression on the zone-years with climate data
cv_rows = []
for target in TARGETS:
    d = targets[(targets.target == target) & targets.zone.isin(ZONES)].reset_index(drop=True)
    cv_rows.append(cv_row(target, "trend", d, loyo_predict(d, lambda tr: ["year"])))
for (source, target), fm in final_models.items():
    feats = candidate_features(source, fm["data"])
    cv_rows.append(cv_row(target, "climate", fm["data"],
                          loyo_predict(fm["data"], lambda tr: select_features(tr, feats))))

# ---- CH2025: the final regression of 3b per model chain; 30-year mean per warming level
proj_rows, extrap_rows, chain_means = [], [], []
for (source, target), fm in final_models.items():
    c = climate[source]
    if c["features_scen"] is None:
        print(f"{target} ({source}): no scenarios, no projection")
        continue
    per_chain = regression_projection(fm["model"], c["features_scen"]).mean("year")
    per_chain = xr.concat([per_chain, (per_chain * xr.DataArray(ZONE_SHARE)).sum("zone", min_count=1)
                           .expand_dims(zone=["AOI"])], "zone")
    delta = per_chain - per_chain.sel(period=REFERENCE)
    for period in per_chain.period.values:
        for zone in per_chain.zone.values:
            v = per_chain.sel(period=period, zone=zone).values
            dv = delta.sel(period=period, zone=zone).values
            if np.isfinite(v).sum() == 0:
                continue
            proj_rows.append({"source": source, "target": target, "period": period, "zone": zone,
                              "doy_median": np.nanmedian(v), "doy_p05": np.nanpercentile(v, 5),
                              "doy_p95": np.nanpercentile(v, 95), "delta_median": np.nanmedian(dv),
                              "delta_p05": np.nanpercentile(dv, 5), "delta_p95": np.nanpercentile(dv, 95),
                              "n_chains": int(np.isfinite(v).sum())})
    chain_means.append(per_chain.sel(zone="AOI").to_dataframe(name="doy").reset_index().assign(source=source, target=target))
    for f in fm["model"]["features"]:
        lo, hi = fm["data"][f].min(), fm["data"][f].max()
        sv = c["features_scen"][f]
        outside = ((sv < lo) | (sv > hi)).where(sv.notnull()).mean(["model", "zone", "year"])
        for period, share in outside.to_series().items():
            extrap_rows.append({"source": source, "target": target, "feature": f, "period": period,
                                "share_outside_observed_range": share})
projections_by_gwl, extrapolation = pd.DataFrame(proj_rows), pd.DataFrame(extrap_rows)
chains = (pd.concat(chain_means) if chain_means
          else pd.DataFrame(columns=["target", "source", "period", "doy"]))

# ---- one table: skill and the whole-AOI date per warming level (trend: at the middle of the level's period)
future = {(target, "trend"): dict(zip(PERIODS, trend_line(target, np.array([MID_YEAR[p] for p in PERIODS]))[0]))
          for target in TARGETS}
for (target, source, period), v in chains.groupby(["target", "source", "period"]).doy.median().items():
    future.setdefault((target, "climate"), {})[period] = v
prediction_comparison = (pd.DataFrame(cv_rows).set_index(["target", "approach"])
                         .join(pd.DataFrame(future).T.rename_axis(["target", "approach"]).reindex(columns=PERIODS)))
print("Leave-one-year-out RMSE (days) and skill; whole-AOI date (DOY) per warming level")
prediction_comparison = prediction_comparison.loc[sorted(prediction_comparison.index, key=lambda k: TARGETS.index(k[0]))]
display(prediction_comparison.round({"cv_rmse": 1, "skill": 2, **{p: 0 for p in PERIODS}}))

In [ ]:
GWL_LABELS = {p: "ref" if p == REFERENCE else f"+{p[3:]} °C" for p in PERIODS}
YEARS = np.arange(GWL_YEARS[PERIODS[0]][0], GWL_YEARS[PERIODS[-1]][1] + 1)
skill = prediction_comparison.skill
fig, axes = plt.subplots(1, len(TARGETS), figsize=(5.5 * len(TARGETS), 4.5), squeeze=False)
for ax, target in zip(axes[0], TARGETS):
    a = targets[(targets.target == target) & (targets.zone == "all")]
    fit, lo, hi = trend_line(target, YEARS)
    ax.fill_between(YEARS, lo, hi, color="grey", alpha=0.25, lw=0)
    ax.plot(YEARS, fit, color="grey", lw=1.5, label=f"trend, 95 % CI (skill {skill.get((target, 'trend'), np.nan):+.2f})")
    values = list(a.doy)
    ct = chains[chains.target == target]
    sources = sorted(set(ct.source))
    for k, source in enumerate(sources):
        g = ct[ct.source == source]
        for p in PERIODS:
            v = g[g.period == p].doy.dropna()
            if v.empty:
                continue
            m, q05, q95 = v.median(), v.quantile(0.05), v.quantile(0.95)
            ax.errorbar(MID_YEAR[p] + (k - (len(sources) - 1) / 2) * 3, m, yerr=[[m - q05], [q95 - m]], fmt="o",
                        color=SOURCE_COLORS[source], capsize=3)
            values += [q05, q95]
        ax.plot([], [], "o", color=SOURCE_COLORS[source],
                label=f"regression + CH2025, 5–95 % of chains (skill {skill.get((target, 'climate'), np.nan):+.2f})")
    ax.plot(a.year, a.doy, "ko", ms=4, label="observed")
    ax.set_xlim(YEARS[0] - 3, YEARS[-1] + 3)
    ax.set_ylim(min(values) - 10, max(values) + 10)
    ax.set_xlabel("year")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7, loc="lower left")
    top = ax.twiny()   # warming levels at the middle of their period
    top.set_xlim(ax.get_xlim())
    top.set_xticks([MID_YEAR[p] for p in PERIODS], [GWL_LABELS[p] for p in PERIODS], fontsize=8)
    top.set_title(target)
axes[0, 0].set_ylabel("day of year (whole AOI)")
fig.suptitle("Trend of the observed years vs regression + CH2025 (30-year mean per model chain)")
plt.tight_layout()
plt.show()

## 5. Export

In [ ]:
outputs = {
    "climate_features_obs.csv": pd.concat([f.assign(source=s) for s, f in obs_features.items()]),
    "feature_info.csv": feature_info,
    "climate_effects.csv": climate_effects,
    "climate_regression.csv": climate_regression,
    "prediction_comparison.csv": prediction_comparison.reset_index(),
    "projections_by_gwl.csv": projections_by_gwl,
    "extrapolation.csv": extrapolation,
}
for name, frame in outputs.items():
    frame.to_csv(CLIMATE_OUT_DIR / name, index=False)
print(f"Results written to {CLIMATE_OUT_DIR}")

## References

Studies cited for the features (section 2):

- Asam, S. et al. (2018). Relationship between spatiotemporal variations of climate, snow cover and plant phenology over the Alps: an Earth observation-based analysis. *Remote Sensing* 10, 1757.
- Bigler, C. & Vitasse, Y. (2021). Premature leaf discoloration of European deciduous trees is caused by drought and heat in late spring and cold spells in early fall. *Agricultural and Forest Meteorology*.
- Busetto, L. et al. (2010). Remote sensing of larch phenological cycle and analysis of relationships with climate in the Alpine region. *Global Change Biology* 16, 2504–2517.
- Delpierre, N. et al. (2009). Modelling interannual and spatial variability of leaf senescence for three deciduous tree species in France. *Agricultural and Forest Meteorology* 149, 938–948.
- Estrella, N. & Menzel, A. (2006). Responses of leaf colouring in four deciduous tree species to climate and weather in Germany. *Climate Research* 32, 253–267.
- Liu, Q., Fu, Y. H., Zeng, Z., Huang, M., Li, X. & Piao, S. (2016). Temperature, precipitation, and insolation effects on autumn vegetation phenology in temperate China. *Global Change Biology* 22, 644–655.
- Wu, C. et al. (2018). Contrasting responses of autumn-leaf senescence to daytime and night-time warming. *Nature Climate Change* 8, 1092–1096.
- Xie, Y., Wang, X., Wilson, A. M. & Silander, J. A. (2018). Predicting autumn phenology: how deciduous tree species respond to weather stressors. *Agricultural and Forest Meteorology* 250–251, 127–137.